# LangGraph Tutorial — Beginner to Advanced

A hands-on notebook for learning **LangGraph from first principles**.

## Roadmap

### Beginner
- What LangGraph is
- State, nodes, edges
- `START` and `END`
- Graph compilation and invocation
- Visualization

### Intermediate
- Conditional edges
- Loops
- Reducers
- `MessagesState`
- Streaming

### Agentic
- LLM nodes
- Tools
- `ToolNode`
- `tools_condition`
- ReAct-style agent loops

### Advanced
- Checkpointing
- Threads
- Human-in-the-loop
- `interrupt`
- `Command`
- Subgraphs
- Parallel branches
- Recursion limits
- Functional API
- Production patterns
- Final project

> The first half is completely LLM-free so you can understand LangGraph mechanics before adding model APIs.


## 0. Mental model

Think of LangGraph as:

```text
Graph = State + Nodes + Edges
```

- **State** = shared data
- **Node** = a Python function that performs work
- **Edge** = determines what runs next

A graph can branch, loop, pause, resume, stream, and persist state.


## 1. Installation

In [ ]:
%pip install -q -U langgraph langchain langchain-openai

# Optional for local Ollama models:
# %pip install -q -U langchain-ollama


# LEVEL 1 — Beginner

## 2. Define State

In [ ]:
from typing_extensions import TypedDict

class SimpleState(TypedDict):
    message: str


The state is the shared data structure flowing through the graph.

A node receives the current state and returns **updates** to that state.


## 3. Create a Node

In [ ]:
def hello_node(state: SimpleState):
    print("Received:", state)
    return {"message": state["message"] + " -> Hello from LangGraph!"}


## 4. Build Your First Graph

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(SimpleState)
builder.add_node("hello", hello_node)

builder.add_edge(START, "hello")
builder.add_edge("hello", END)

graph = builder.compile()


Architecture:

```text
START
  ↓
hello
  ↓
 END
```


## 5. Invoke the Graph

In [ ]:
result = graph.invoke({"message": "User input"})
result


## 6. Multiple Nodes

In [ ]:
class PipelineState(TypedDict):
    text: str

def clean_text(state: PipelineState):
    return {"text": state["text"].strip().lower()}

def add_prefix(state: PipelineState):
    return {"text": "Processed: " + state["text"]}

builder = StateGraph(PipelineState)
builder.add_node("clean", clean_text)
builder.add_node("prefix", add_prefix)

builder.add_edge(START, "clean")
builder.add_edge("clean", "prefix")
builder.add_edge("prefix", END)

pipeline = builder.compile()

pipeline.invoke({"text": "   HELLO LANGGRAPH   "})


## 7. Visualize a Graph

In [ ]:
from IPython.display import Image, display

try:
    display(Image(pipeline.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Visualization is optional; graph execution still works.")
    print(type(e).__name__, e)


# LEVEL 2 — Intermediate

## 8. Conditional Edges

In [ ]:
from typing import Literal

class SentimentState(TypedDict):
    text: str
    sentiment: str
    response: str

def classify(state: SentimentState):
    positive_words = {"great", "good", "excellent", "love", "happy"}
    words = set(state["text"].lower().split())
    sentiment = "positive" if words & positive_words else "negative"
    return {"sentiment": sentiment}

def positive_response(state: SentimentState):
    return {"response": "Glad to hear that!"}

def negative_response(state: SentimentState):
    return {"response": "I understand. Let's see how we can improve it."}

def route_sentiment(state: SentimentState) -> Literal["positive", "negative"]:
    return "positive" if state["sentiment"] == "positive" else "negative"

builder = StateGraph(SentimentState)
builder.add_node("classify", classify)
builder.add_node("positive", positive_response)
builder.add_node("negative", negative_response)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route_sentiment)
builder.add_edge("positive", END)
builder.add_edge("negative", END)

sentiment_graph = builder.compile()


In [ ]:
print(sentiment_graph.invoke({
    "text": "I love LangGraph",
    "sentiment": "",
    "response": ""
}))

print(sentiment_graph.invoke({
    "text": "This is difficult",
    "sentiment": "",
    "response": ""
}))


Conditional routing lets execution choose a path dynamically:

```text
             ┌→ positive → END
classify ────┤
             └→ negative → END
```


## 9. Loops

In [ ]:
class CounterState(TypedDict):
    count: int

def increment(state: CounterState):
    print("count =", state["count"])
    return {"count": state["count"] + 1}

def continue_or_stop(state: CounterState):
    if state["count"] < 5:
        return "increment"
    return END

builder = StateGraph(CounterState)
builder.add_node("increment", increment)

builder.add_edge(START, "increment")
builder.add_conditional_edges("increment", continue_or_stop)

counter_graph = builder.compile()

counter_graph.invoke({"count": 0})


Loops are central to agent systems.

Always define a termination condition:

```text
increment → condition
   ↑          │
   └── loop ──┘
              └→ END
```


## 10. Reducers

In [ ]:
import operator
from typing import Annotated

class ReducerState(TypedDict):
    history: Annotated[list[str], operator.add]

def step_a(state: ReducerState):
    return {"history": ["A"]}

def step_b(state: ReducerState):
    return {"history": ["B"]}

builder = StateGraph(ReducerState)
builder.add_node("a", step_a)
builder.add_node("b", step_b)

builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", END)

reducer_graph = builder.compile()

reducer_graph.invoke({"history": []})


Without a reducer, updates normally overwrite a key.

With:

```python
Annotated[list[str], operator.add]
```

new list values are accumulated.


## 11. MessagesState

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, AIMessage

class CustomMessagesState(MessagesState):
    user_id: str
    num_llm_calls: int

example = {
    "messages": [
        HumanMessage(content="Hello"),
        AIMessage(content="Hi!")
    ],
    "user_id": "user-1",
    "num_llm_calls": 1
}

example


`MessagesState` is commonly used for chat and agent applications because it already defines a message-aware reducer.


## 12. Streaming

In [ ]:
class StreamState(TypedDict):
    value: int

def double(state: StreamState):
    return {"value": state["value"] * 2}

def add_ten(state: StreamState):
    return {"value": state["value"] + 10}

builder = StateGraph(StreamState)
builder.add_node("double", double)
builder.add_node("add_ten", add_ten)

builder.add_edge(START, "double")
builder.add_edge("double", "add_ten")
builder.add_edge("add_ten", END)

stream_graph = builder.compile()

for event in stream_graph.stream({"value": 5}):
    print(event)


# LEVEL 3 — Agentic LangGraph

## 13. Configure an LLM

### OpenAI option

Uncomment the next cell after setting `OPENAI_API_KEY`.


In [ ]:
# from langchain_openai import ChatOpenAI
#
# model = ChatOpenAI(
#     model="gpt-4.1-mini",
#     temperature=0,
# )


### Ollama option

In [ ]:
# from langchain_ollama import ChatOllama
#
# model = ChatOllama(
#     model="llama3.1:8b",
#     temperature=0,
# )


Choose one provider so that a variable named `model` exists before running the following LLM sections.


## 14. Minimal LLM Graph

In [ ]:
# class LLMState(MessagesState):
#     pass
#
# def call_model(state: LLMState):
#     response = model.invoke(state["messages"])
#     return {"messages": [response]}
#
# builder = StateGraph(LLMState)
# builder.add_node("llm", call_model)
# builder.add_edge(START, "llm")
# builder.add_edge("llm", END)
#
# llm_graph = builder.compile()
#
# result = llm_graph.invoke({
#     "messages": [
#         HumanMessage(content="Explain LangGraph in one sentence.")
#     ]
# })
#
# print(result["messages"][-1].content)


## 15. Define Tools

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

tools = [multiply, add]

multiply.invoke({"a": 6, "b": 7})


## 16. Bind Tools to the Model

In [ ]:
# model_with_tools = model.bind_tools(tools)


## 17. ToolNode and tools_condition

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

tool_node = ToolNode(tools)


Core loop:

```text
        ┌───────────────────┐
        ↓                   │
      MODEL ──tool call──→ TOOLS
        │                   │
        └──no tool call→ END
```


## 18. Build a ReAct-style Agent

In [ ]:
# def agent_node(state: MessagesState):
#     response = model_with_tools.invoke(state["messages"])
#     return {"messages": [response]}
#
# builder = StateGraph(MessagesState)
# builder.add_node("agent", agent_node)
# builder.add_node("tools", ToolNode(tools))
#
# builder.add_edge(START, "agent")
# builder.add_conditional_edges("agent", tools_condition)
# builder.add_edge("tools", "agent")
#
# agent_graph = builder.compile()
#
# result = agent_graph.invoke({
#     "messages": [
#         HumanMessage(content="What is 19 multiplied by 23? Use a tool.")
#     ]
# })
#
# for msg in result["messages"]:
#     print(type(msg).__name__, ":", msg.content)


This is a fundamental agent pattern:

```text
Reason → Act → Observe → Reason → ... → Answer
```

LangGraph makes the loop explicit and controllable.


# LEVEL 4 — Advanced

## 19. Checkpointing

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()


A checkpointer saves graph state snapshots.

Use in-memory persistence for learning/testing. Use a durable backend such as PostgreSQL for production.


## 20. Threads

In [ ]:
# Rebuild the tool agent with persistence after configuring `model`.
#
# builder = StateGraph(MessagesState)
# builder.add_node("agent", agent_node)
# builder.add_node("tools", ToolNode(tools))
#
# builder.add_edge(START, "agent")
# builder.add_conditional_edges("agent", tools_condition)
# builder.add_edge("tools", "agent")
#
# persistent_agent = builder.compile(checkpointer=memory)
#
# config = {"configurable": {"thread_id": "conversation-1"}}
#
# persistent_agent.invoke(
#     {"messages": [HumanMessage(content="My favorite number is 42.")]},
#     config=config,
# )
#
# result = persistent_agent.invoke(
#     {"messages": [HumanMessage(content="What is my favorite number?")]},
#     config=config,
# )
#
# print(result["messages"][-1].content)


Reusing the same `thread_id` continues the same persisted workflow state.

Important:

```text
Checkpointer → thread-scoped workflow state
Store        → cross-thread long-term application memory
```


## 21. Human-in-the-loop with interrupt()

In [ ]:
from langgraph.types import interrupt, Command

class ApprovalState(TypedDict):
    action: str
    approved: bool

def approval_node(state: ApprovalState):
    decision = interrupt({
        "question": "Do you approve this action?",
        "action": state["action"],
    })
    return {"approved": bool(decision)}

builder = StateGraph(ApprovalState)
builder.add_node("approval", approval_node)
builder.add_edge(START, "approval")
builder.add_edge("approval", END)

approval_graph = builder.compile(checkpointer=InMemorySaver())

approval_config = {
    "configurable": {
        "thread_id": "approval-demo"
    }
}

paused = approval_graph.invoke(
    {
        "action": "Delete 100 database rows",
        "approved": False
    },
    config=approval_config
)

paused


## 22. Resume with Command

In [ ]:
resumed = approval_graph.invoke(
    Command(resume=True),
    config=approval_config
)

resumed


Important: when resuming an interrupt, the node containing `interrupt()` starts again from the beginning. Side effects before an interrupt should therefore be designed to be idempotent.


## 23. Command for State Updates + Routing

In [ ]:
from typing import Literal

class CommandState(TypedDict):
    score: int
    result: str

def judge(state: CommandState) -> Command[Literal["pass_node", "fail_node"]]:
    if state["score"] >= 70:
        return Command(
            update={"result": "passed"},
            goto="pass_node"
        )
    return Command(
        update={"result": "failed"},
        goto="fail_node"
    )

def pass_node(state: CommandState):
    print("PASS")
    return {}

def fail_node(state: CommandState):
    print("FAIL")
    return {}

builder = StateGraph(CommandState)
builder.add_node("judge", judge)
builder.add_node("pass_node", pass_node)
builder.add_node("fail_node", fail_node)

builder.add_edge(START, "judge")
builder.add_edge("pass_node", END)
builder.add_edge("fail_node", END)

command_graph = builder.compile()

command_graph.invoke({"score": 85, "result": ""})


## 24. Subgraphs

In [ ]:
class SharedState(TypedDict):
    text: str

# Child graph
def lowercase(state: SharedState):
    return {"text": state["text"].lower()}

sub_builder = StateGraph(SharedState)
sub_builder.add_node("lowercase", lowercase)
sub_builder.add_edge(START, "lowercase")
sub_builder.add_edge("lowercase", END)
text_subgraph = sub_builder.compile()

# Parent graph
def add_label(state: SharedState):
    return {"text": "[DONE] " + state["text"]}

parent_builder = StateGraph(SharedState)
parent_builder.add_node("text_processing", text_subgraph)
parent_builder.add_node("label", add_label)

parent_builder.add_edge(START, "text_processing")
parent_builder.add_edge("text_processing", "label")
parent_builder.add_edge("label", END)

parent_graph = parent_builder.compile()

parent_graph.invoke({"text": "HELLO LANGGRAPH"})


Subgraphs are useful for reusable workflows, specialist agents, and multi-agent systems.


## 25. Parallel Branches

In [ ]:
class ParallelState(TypedDict):
    results: Annotated[list[str], operator.add]

def source(state: ParallelState):
    return {"results": ["source"]}

def branch_a(state: ParallelState):
    return {"results": ["A"]}

def branch_b(state: ParallelState):
    return {"results": ["B"]}

builder = StateGraph(ParallelState)
builder.add_node("source", source)
builder.add_node("branch_a", branch_a)
builder.add_node("branch_b", branch_b)

builder.add_edge(START, "source")
builder.add_edge("source", "branch_a")
builder.add_edge("source", "branch_b")
builder.add_edge("branch_a", END)
builder.add_edge("branch_b", END)

parallel_graph = builder.compile()

parallel_graph.invoke({"results": []})


## 26. Recursion Limits

In [ ]:
from langgraph.errors import GraphRecursionError

class InfiniteState(TypedDict):
    n: int

def forever(state: InfiniteState):
    return {"n": state["n"] + 1}

builder = StateGraph(InfiniteState)
builder.add_node("forever", forever)
builder.add_edge(START, "forever")
builder.add_edge("forever", "forever")

infinite_graph = builder.compile()

try:
    infinite_graph.invoke(
        {"n": 0},
        config={"recursion_limit": 5}
    )
except GraphRecursionError:
    print("Stopped: recursion limit reached.")


For any loop, ask:

**What causes execution to terminate?**

Typical answers:
- no tool call
- maximum retries
- confidence threshold
- enough evidence retrieved
- explicit completion state
- human approval/rejection


## 27. Functional API

In [ ]:
from langgraph.func import entrypoint, task

@task
def multiply_task(x: int) -> int:
    return x * 2

@task
def add_task(x: int) -> int:
    return x + 10

@entrypoint()
def functional_workflow(x: int):
    doubled = multiply_task(x).result()
    final = add_task(doubled).result()
    return final

functional_workflow.invoke(5)


The Graph API and Functional API use the same runtime.

Use the **Graph API** when explicit graph topology helps.

Use the **Functional API** when the workflow naturally resembles normal Python control flow.


# 28. Common Agent Architecture Patterns

## Router

```text
                 ┌→ technical_agent
User → router ───┼→ billing_agent
                 └→ general_agent
```

## ReAct

```text
       ┌─────────────────┐
       ↓                 │
     Agent → Tool → Observation
       │
       └──── answer → END
```

## Reflection

```text
generate → critique → good enough?
                      ↙        ↘
                    yes         no
                     ↓           │
                    END ← revise ┘
```

## Human approval

```text
agent → risky action? → interrupt → human
                               ↙       ↘
                            approve   reject
```

## Supervisor + specialists

```text
                 ┌→ Research Agent
User → Supervisor├→ Coding Agent
                 └→ Data Agent
```


# 29. Final Project — Stateful Research Assistant

Architecture:

```text
                         ┌────────────────────┐
                         │                    │
                         ▼                    │
User → Agent → Need tool? ─yes→ ToolNode ────┘
                 │
                 no
                 ↓
                END
```

Features:
- message state
- tool calling
- conditional routing
- iterative tool use
- checkpointing
- thread memory
- streaming


In [ ]:
# Uncomment after configuring `model`.
#
# @tool
# def knowledge_search(query: str) -> str:
#     """Search a tiny demo knowledge base."""
#     knowledge = {
#         "langgraph": "LangGraph orchestrates stateful workflows and agents.",
#         "langchain": "LangChain provides LLM application abstractions and integrations.",
#         "rag": "RAG retrieves external information before generation."
#     }
#
#     q = query.lower()
#     for key, value in knowledge.items():
#         if key in q:
#             return value
#     return "No matching information was found."
#
# research_tools = [knowledge_search]
# research_model = model.bind_tools(research_tools)
#
# class ResearchState(MessagesState):
#     llm_calls: int
#
# def research_agent(state: ResearchState):
#     response = research_model.invoke(state["messages"])
#     return {
#         "messages": [response],
#         "llm_calls": state.get("llm_calls", 0) + 1
#     }
#
# builder = StateGraph(ResearchState)
# builder.add_node("agent", research_agent)
# builder.add_node("tools", ToolNode(research_tools))
#
# builder.add_edge(START, "agent")
# builder.add_conditional_edges("agent", tools_condition)
# builder.add_edge("tools", "agent")
#
# research_graph = builder.compile(
#     checkpointer=InMemorySaver()
# )


In [ ]:
# config = {
#     "configurable": {
#         "thread_id": "research-session-1"
#     }
# }
#
# result = research_graph.invoke(
#     {
#         "messages": [
#             HumanMessage(
#                 content="What is LangGraph? Use the knowledge tool."
#             )
#         ],
#         "llm_calls": 0
#     },
#     config=config
# )
#
# print(result["messages"][-1].content)
# print("LLM calls:", result["llm_calls"])


## 30. Stream the Final Agent

In [ ]:
# for event in research_graph.stream(
#     {
#         "messages": [
#             HumanMessage(content="Explain LangChain and LangGraph.")
#         ],
#         "llm_calls": 0
#     },
#     config={
#         "configurable": {
#             "thread_id": "stream-example"
#         }
#     }
# ):
#     print(event)


# 31. Production Persistence

For learning:

```text
InMemorySaver
```

For local/small persistent projects:

```text
SQLite checkpointer
```

For production:

```text
PostgreSQL checkpointer
```

Persistence supports:
- recovery after failures
- thread continuity
- pause/resume
- human-in-the-loop
- state history
- durable workflows


# 32. LangChain vs LangGraph

```text
LANGCHAIN
────────────────────────
Models
Prompts
Tools
Retrievers
Vector stores
Structured output
High-level agents
Integrations

          │
          ▼

LANGGRAPH
────────────────────────
State
Nodes
Edges
Conditional routing
Loops
Persistence
Interrupts
Subgraphs
Durable execution
```

Useful mental model:

```text
LangChain = components + high-level abstractions
LangGraph = orchestration + state + execution control
```


# 33. Exercises

### Exercise 1
Build:

```text
START → multiply_by_2 → subtract_3 → END
```

Input `10`, expected output `17`.

### Exercise 2
Build age routing:

```text
            ┌→ adult
age_check ──┤
            └→ minor
```

### Exercise 3
Loop from `count=0` until `count=10`.

### Exercise 4
Use a reducer to produce:

```python
["node_1", "node_2", "node_3"]
```

### Exercise 5
Give an LLM two tools:
- calculator
- fake weather

Let the model decide which to call.

### Exercise 6
Create a `delete_file` workflow that pauses for human approval.

### Exercise 7 — Advanced
Build:

```text
                    ┌→ vector_search ─┐
User → router ──────┤                 ├→ answer
                    └→ web_search ────┘
                              ↓
                           critique
                              ↓
                        good enough?
                         ↙        ↘
                       yes         no
                        ↓           │
                       END ←────────┘
```

Add persistence, maximum retries, streaming, and human approval for risky actions.


# 34. Common Mistakes

1. **Mutating state globally** instead of returning explicit updates.
2. **Loops without termination conditions.**
3. Putting the whole application inside one giant node.
4. Using LangGraph for a trivial one-step LLM call.
5. Confusing checkpoint state with long-term cross-thread memory.
6. Performing irreversible side effects before `interrupt()` without considering node replay.


# 35. Recommended Learning Path

```text
LangGraph fundamentals
        ↓
Tool-calling agents
        ↓
Persistence
        ↓
Human-in-the-loop
        ↓
RAG with LangGraph
        ↓
Agentic RAG
        ↓
Multi-agent systems
        ↓
LangSmith tracing/evaluation
        ↓
MCP / external tools
        ↓
Production deployment
```

If you can build the final exercises without copying the earlier cells, you have a strong working understanding of LangGraph.
